In [ ]:
# --- INSTALACIÓN DE LIBRERÍAS PARA YOLOv12 ---
!pip install -q git+https://github.com/sunsmarterjie/yolov12.git
!pip install -q kagglehub

import kagglehub
import os
from ultralytics import YOLO
import torch

print(f" Pytorch version: {torch.__version__}")
print(f" GPU Disponible: {torch.cuda.is_available()}")

In [ ]:
# --- DESCARGA DEL DATASET ---
print(" Descargando dataset de Kaggle...")
dataset_path = kagglehub.dataset_download("nikolasgegenava/sard-2-search-and-rescue-dataset-extra-classes")

print(f" Dataset descargado en: {dataset_path}")

# --- BÚSQUEDA AUTOMÁTICA DEL YAML ---
yaml_path = None
for root, dirs, files in os.walk(dataset_path):
    if 'data.yaml' in files:
        yaml_path = os.path.join(root, 'data.yaml')
        break

if yaml_path:
    print(f" Archivo de configuración encontrado en: {yaml_path}")
else:
    print(" ERROR: No se encontró 'data.yaml'.")

In [ ]:
# --- ENTRENAMIENTO CON YOLOv12 ---

# 1. Cargar el modelo YOLOv12.
print(" Cargando modelo YOLOv12s...")
model = YOLO('yolov12s.pt')

# 2. Configurar Hiperparámetros y Entrenar
print(" Iniciando entrenamiento de YOLOv12...")

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    project='runs/detect',
    name='yolov12_sard_rescue',
    exist_ok=True,
    plots=True
)

print(" Entrenamiento finalizado.")

In [ ]:
import shutil

# Ruta donde Ultralytics guarda el mejor modelo
source_best = os.path.join('runs', 'detect', 'yolov12_sard_rescue', 'weights', 'best.pt')
dest_folder = '/content/drive/MyDrive/TFG'
dest_file = os.path.join(dest_folder, 'best_yolov12.pt')

# Montar Drive
from google.colab import drive
drive.mount('/content/drive')

if os.path.exists(source_best):
    print(f" Guardando mejor modelo en: {dest_file}")
    if not os.path.exists(dest_folder):
        os.makedirs(dest_folder, exist_ok=True)

    shutil.copy(source_best, dest_file)
    print(" ¡Modelo guardado con éxito en Google Drive!")
else:
    print(f" No se encontró el archivo {source_best}")

In [ ]:
from google.colab import drive
import os
import shutil

# Intentamos montar con force_remount primero
try:
    drive.mount('/content/drive', force_remount=True)
    print(" Drive montado correctamente.")
except ValueError:
    # Si falla porque la carpeta tiene archivos locales, la limpiamos (hacemos backup por seguridad)
    print(" La carpeta /content/drive tiene archivos locales. Solucionando...")

    # Si existe y no es un montaje real, la movemos para no perder datos por si acaso
    if os.path.exists('/content/drive') and not os.path.ismount('/content/drive'):
        os.rename('/content/drive', '/content/drive_backup_conflict')
        print("   -> Carpeta conflictiva renombrada a '/content/drive_backup_conflict'")

        # Creamos la carpeta limpia y montamos
        os.makedirs('/content/drive')
        drive.mount('/content/drive')
        print(" Drive montado tras limpiar conflicto.")